In [1]:
!pip install datasets
!pip install transformers
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install torch


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Hugging face Squads Dataset
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("rajpurkar/squad")

In [3]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [4]:
# samples from the dataset
sample = ds["train"][0]

print("CONTEXT:\n")
print(sample["context"])

print("\nQUESTION:\n")
print(sample["question"])

print("\nANSWER:\n")
print(sample["answers"]["text"][0])

CONTEXT:

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

QUESTION:

To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

ANSWER:

Saint Bernadette Soubirous


In [5]:
# Load FAISS vector database
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [6]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

C:\Users\KAUSHIK\AppData\Local\Temp\ipykernel_15712\2733826177.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [7]:
# Load LLM
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

In [8]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
    temperature=0.2,
    repetition_penalty=1.2
)

llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
C:\Users\KAUSHIK\AppData\Local\Temp\ipykernel_15712\3715024748.py:9: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [9]:
# Build retrieval function
def retrieve_context(topic):

    docs = db.max_marginal_relevance_search(
        topic,
        k=3,
        fetch_k=10
    )

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return context[:1500]

In [10]:
context = retrieve_context("clustering")

print(context)

Clustering Algorithms
1 Introduction to Clustering
Clustering is anunsupervised learningtechnique that aims to group a set of data
objects into clusters such that objects within the same cluster are more similar to each
other than to those in other clusters. Similarity is usually measured using distance metrics
such as Euclidean, Manhattan, or cosine distance.
1.1 Objectives of Clustering
Clustering aims to organize unlabeled data into meaningful groups based on similarity

4 Partitioning Around Medoids (PAM)
4.1 Overview
Partitioning Around Medoids (PAM) is a classical and widely studied algorithm for solv-
ing the K-medoids clustering problem. Unlike K-means, which represents clusters using
centroids that may not correspond to actual data points, PAM identifies representative
objects calledmedoidsfrom the dataset itself. The algorithm explicitly searches for an
optimal set of medoids by minimizing the total dissimilarity between data points and

computer vision.
•Bioinformatics and g

In [11]:
# Build question generation function
def generate_questions(topic):

    context = retrieve_context(topic)

    prompt = f"""
You are an expert educational AI tutor.

Your task is to generate HIGH-QUALITY educational questions
from the provided context.

Generate:
1. Two Multiple Choice Questions (MCQ)
   - each with 4 options
   - mention correct answer

2. Two Short Answer Questions

3. One Long Answer Question

IMPORTANT RULES:
- Questions must be specific to the context
- Avoid generic questions
- Do NOT repeat questions
- Focus on concepts, architecture, mechanisms, and reasoning

Context:
{context}

Output Format:

MCQ 1:
Question:
Options:
A.
B.
C.
D.
Correct Answer:

MCQ 2:
...

Short Answer 1:
...

Short Answer 2:
...

Long Answer:
...
"""

    response = llm.invoke(prompt)

    return response

In [12]:
# Test the question generation function
questions = generate_questions(
    "clustering in machine learning"
)

print(questions)

Which of the following is not a characteristic of hierarchical clustering?
